# Laufnummer-Korrektur (axon_logger)

Fuehrt **nur** den Laufnummer-Korrektur-Schritt aus `py/src/laufnummer_correction.py`
(`correct_session_folder`) aus, fuer alle drei Hermle-Sessions.

**Was das macht:** liest die rohen `axon_logger_ptw_LevNNN.raw.json`-Dumps je Session
(`<session>/raw/`, falls vorhanden, sonst direkt `<session>/`), erkennt echte
Laufwechsel anhand von Dokument-Identitaet (`_id.$oid`) und Aenderungen der rohen
Laufnummer (`workpiecedata[0].serialnr[3]`) in zeitlicher Reihenfolge, und schreibt
korrigierte Kopien (gleicher Dateiname) nach `<session>/prep/`. Hintergrund und
bekannte Fehlerbilder (mehrere Laeufe in einer Datei, wiederverwendete/uebersprungene
Nummern, nicht automatisch trennbare Merges) siehe Modul-Docstring von
`laufnummer_correction.py`.

**Was das NICHT macht:** das anschliessende Aufteilen der korrigierten Daten in je
eine Datei pro Laufnummer (`split_by_laufnummer()`, Ausgabe nach
`<session>/separated_by_ln/`) ist ein separater Schritt und wird hier nicht
ausgefuehrt.

**Laufzeit:** liest/schreibt alle Rohdaten je Session (bis zu mehreren GB je Datei,
insgesamt ca. 19 GB ueber alle drei Sessions) -- ein voller Durchlauf kann grob
15-30 Minuten dauern.

**Vertraulichkeitshinweis:** siehe `cnc_cut_extractor.py` -- dieselben Rohdaten
(Maschinen-, Prozess- und Werkstueckdaten der PTW-Versuchsreihe). Bitte Notebook-
Outputs (Pfade, Seriennummern) vor externer Weitergabe pruefen.

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "py" / "src"))
from laufnummer_correction import correct_session_folder, resolve_input_output

pd.set_option("display.width", 140)

## 1 - Sessions konfigurieren

In [ ]:
DATA_ROOT = Path.cwd().parent / "data" / "Hermle"
SESSIONS = ["20260630_ALU_y", "20260701_ALU_x", "20260701_C45_y"]

for session in SESSIONS:
    in_dir, out_dir = resolve_input_output(DATA_ROOT / session)
    print(f"{session}: {in_dir} -> {out_dir}")

## 2 - Korrektur je Session ausfuehren

`resolve_input_output()` nutzt `<session>/raw/` als Quelle, falls vorhanden (sonst
`<session>/` direkt), und schreibt immer nach `<session>/prep/`. Warnungen (z.B.
mehrdeutige Baseline, fehlende Laufnummer) werden pro Session eingesammelt und
ausgegeben, statt den Lauf abzubrechen.

In [ ]:
results = {}

for session in SESSIONS:
    session_dir = DATA_ROOT / session
    in_dir, out_dir = resolve_input_output(session_dir)
    print(f"=== {session}: {in_dir} -> {out_dir} ===")

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        file_summary, group_summary = correct_session_folder(in_dir, out_dir)
        for w in caught:
            print(f"  WARNUNG: {w.message}")

    file_summary_path = out_dir / f"laufnummer_correction_file_summary_{session}.csv"
    group_summary_path = out_dir / f"laufnummer_correction_group_summary_{session}.csv"
    file_summary.to_csv(file_summary_path, index=False)
    group_summary.to_csv(group_summary_path, index=False)
    print(f"Zusammenfassungen gespeichert: {file_summary_path}, {group_summary_path}\n")

    results[session] = {"file_summary": file_summary, "group_summary": group_summary}

## 3 - Ergebnisse pruefen

`file_summary`: eine Zeile je verarbeiteter LevNNN-Datei (Anzahl Dokumente, Anzahl
neuer Dokumente, erkannte Tag-Wechsel, Zaehlerstand danach).

`group_summary`: eine Zeile je korrigierter Laufnummer (Dokumentanzahl). `anomalous
== True` markiert Gruppen, die deutlich groesser sind als der Median -- moeglicher
Hinweis auf einen nicht automatisch trennbaren Merge (wie der bekannte C45_y-Tag-28-
Fall). Diese Gruppen werden bewusst NICHT automatisch aufgeteilt und sollten manuell
geprueft werden.

In [ ]:
for session, r in results.items():
    print(f"=== {session}: file_summary ===")
    display(r["file_summary"])

In [ ]:
for session, r in results.items():
    print(f"=== {session}: group_summary ===")
    display(r["group_summary"])
    n_anom = int(r["group_summary"]["anomalous"].sum()) if not r["group_summary"].empty else 0
    if n_anom:
        print(f"  -> {n_anom} auffaellige Gruppe(n), zur manuellen Pruefung markiert.")